# 02 · Leitura — INMET (estações meteorológicas automáticas)

**Camada 1 do ETL — Leitura.** Baixa os arquivos anuais do INMET, descompacta e grava os dados
horários de todas as estações em Parquet (particionado por ano).

| | |
|---|---|
| **Fonte** | [INMET — Dados Históricos](https://portal.inmet.gov.br/dadoshistoricos) |
| **Entrada** | `data/raw/inmet/zip/{ano}.zip` — um CSV horário por estação (~600 por ano) |
| **Saída** | `data/01_leitura/inmet/horario/ano={ano}/*.parquet` e `estacoes_por_ano.csv` |
| **Próxima etapa** | `02_tratamento/02_tratamento_inmet.ipynb` |

Nesta camada fazemos apenas o necessário para **ler** os arquivos: separar o cabeçalho de
metadados da estação, padronizar nomes de colunas, converter decimal com vírgula e marcar como
ausente o código `-9999` do INMET. Limites físicos, completude e agregações ficam no Tratamento.

In [1]:
import sys
import zipfile
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))

import download_inmet  # noqa: E402
import leitura_inmet  # noqa: E402
from config import ANO_FIM, ANO_INICIO, LEITURA, RAW_INMET, garantir_pastas  # noqa: E402

garantir_pastas()

## 1. Download dos arquivos anuais (idempotente)

Mesmo período do SINAN (2014–2026). Arquivos já baixados com o tamanho correto são pulados.

In [2]:
zips = download_inmet.baixar(ANO_INICIO, ANO_FIM)
pd.DataFrame({"arquivo": [z.name for z in zips], "MB": [round(z.stat().st_size / 1e6, 1) for z in zips]})

[2014] já existe (98.9 MB) — pulando


[2015] já existe (98.0 MB) — pulando


[2016] já existe (102.4 MB) — pulando


[2017] já existe (109.7 MB) — pulando


[2018] já existe (117.9 MB) — pulando


[2019] já existe (117.6 MB) — pulando


... (4 outputs left out)

[2024] já existe (102.8 MB) — pulando


[2025] já existe (90.9 MB) — pulando


[2026] já existe (81.8 MB) — pulando


,arquivo,MB
0,2014.zip,98.9
1,2015.zip,98.0
2,2016.zip,102.4
3,2017.zip,109.7
4,2018.zip,117.9
5,2019.zip,117.6
6,2020.zip,103.7
7,2021.zip,80.6
8,2022.zip,90.4
9,2023.zip,107.1


## 2. Como é o arquivo bruto

Cada CSV traz 8 linhas de metadados da estação, uma linha de cabeçalho e os registros horários
(UTC). **O layout mudou em 2019**: a data passou de `2018-01-01` para `2019/01/01`, a hora de
`00:00` para `0000 UTC` e a ausência de dado passou a ser campo vazio (antes era `-9999`).
A ordem das 17 medições não mudou — por isso elas são mapeadas por posição.

In [3]:
for ano in (2018, 2024):
    with zipfile.ZipFile(RAW_INMET / "zip" / f"{ano}.zip") as z:
        nome = next(n for n in z.namelist() if n.lower().endswith(".csv"))
        linhas = z.read(nome).decode("latin1").splitlines()
    print(f"--- {ano}: {nome}")
    print("\n".join(l[:120] for l in linhas[:11]), "\n")

--- 2018: 2018/INMET_CO_DF_A001_BRASILIA_01-01-2018_A_31-12-2018.CSV
REGIÃO:;CO
UF:;DF
ESTAÇÃO:;BRASILIA
CODIGO (WMO):;A001
LATITUDE:;-15,78944444
LONGITUDE:;-47,92583332
ALTITUDE:;1159,54
DATA DE FUNDAÇÃO (YYYY-MM-DD):;2000-05-07
DATA (YYYY-MM-DD);HORA (UTC);PRECIPITAÇÃO TOTAL, HORÁRIO (mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB);PRES
2018-01-01;00:00;0;887,8;887,8;887,1;-9999;21,8;18;21,8;21,6;18,2;17,7;81;78;79;117;4;1,6;
2018-01-01;01:00;0;888,2;888,3;887,8;-9999;21,8;17,9;21,9;21,5;18,5;17,9;83;78;79;135;2,8;1,3; 

--- 2024: INMET_CO_DF_A001_BRASILIA_01-01-2024_A_31-12-2024.CSV
REGIAO:;CO
UF:;DF
ESTACAO:;BRASILIA
CODIGO (WMO):;A001
LATITUDE:;-15,78944444
LONGITUDE:;-47,92583332
ALTITUDE:;1160,96
DATA DE FUNDACAO:;07/05/00
Data;Hora UTC;PRECIPITAÇÃO TOTAL, HORÁRIO (mm);PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB);PRESSÃO ATMOSFERICA
2024/01/01;0000 UTC;0;885,7;885,7;885,3;;22;18,2;22,6;21,8;18,6;18,2;81;77;79;311;2,8;1,5;
2024/01/01;0100 UTC;0;886,7;886,7;885

## 3. Leitura de todos os anos → Parquet horário

Processamento paralelo (um processo por ano). Leva cerca de 1 minuto.

In [4]:
metas = leitura_inmet.processar(workers=4)
resumo = (metas.groupby("ano")
          .agg(estacoes=("codigo_wmo", "nunique"), registros_horarios=("n_registros", "sum"))
          .reset_index())
resumo

,ano,estacoes,registros_horarios
0,2014,475,4147032
1,2015,484,4191888
2,2016,529,4372128
3,2017,563,4775544
4,2018,596,5128416
5,2019,589,5118216
6,2020,589,5173776
7,2021,588,5150880
8,2022,567,4948920
9,2023,567,4966920


## 4. Checagens da leitura

In [5]:
horario = pd.read_parquet(LEITURA / "inmet" / "horario", filters=[("ano", "=", 2024)])
print(horario.shape)
horario.head()

(4962960, 20)


,codigo_wmo,datahora_utc,precipitacao_mm,pressao_mb,pressao_max_mb,pressao_min_mb,radiacao_kjm2,temp_ar_c,temp_orvalho_c,temp_max_c,temp_min_c,orvalho_max_c,orvalho_min_c,umidade_max_pct,umidade_min_pct,umidade_pct,vento_direcao_graus,vento_rajada_ms,vento_velocidade_ms,ano
0,A001,2024-01-01 00:00:00,0.0,885.700012,885.700012,885.299988,NaN,22.000000,18.200001,22.6,21.799999,18.600000,18.200001,81.0,77.0,79.0,311.0,2.8,1.5,2024
1,A001,2024-01-01 01:00:00,0.0,886.700012,886.700012,885.700012,NaN,21.200001,18.600000,22.0,21.100000,18.700001,18.100000,85.0,79.0,85.0,315.0,2.9,1.5,2024
2,A001,2024-01-01 02:00:00,0.0,887.099976,887.200012,886.700012,NaN,20.900000,18.799999,21.5,20.900000,18.900000,18.700001,88.0,85.0,88.0,305.0,3.3,1.1,2024
3,A001,2024-01-01 03:00:00,0.0,887.099976,887.400024,887.099976,NaN,20.900000,18.299999,21.1,20.799999,18.900000,18.299999,88.0,85.0,85.0,304.0,3.3,1.5,2024
4,A001,2024-01-01 04:00:00,0.0,886.400024,887.099976,886.400024,NaN,20.500000,18.600000,21.0,20.500000,18.600000,18.299999,89.0,85.0,89.0,315.0,4.0,2.0,2024


In [6]:
# Fração de valores ausentes (NaN) por variável, tal como entregue pelo INMET — antes de qualquer limpeza
(horario.drop(columns=["codigo_wmo", "datahora_utc", "ano"]).isna().mean().mul(100).round(1)
 .sort_values(ascending=False).rename("% ausente (2024)").to_frame())

,% ausente (2024)
radiacao_kjm2,49.5
precipitacao_mm,26.3
orvalho_max_c,24.5
orvalho_min_c,24.5
temp_orvalho_c,24.3
umidade_min_pct,24.0
umidade_max_pct,24.0
umidade_pct,23.9
vento_velocidade_ms,23.8
vento_rajada_ms,23.7


In [7]:
# Mesma estação pode mudar levemente de coordenada entre anos (relocação/recalibração do GPS)
variacao = metas.groupby("codigo_wmo")[["latitude", "longitude"]].agg(lambda s: s.max() - s.min())
print(f"{metas.codigo_wmo.nunique()} estações distintas em {len(metas)} arquivos estação-ano; "
      f"{(variacao.max(axis=1) > 0.01).sum()} mudaram de coordenada em mais de 0,01° no período.")

705 estações distintas em 7340 arquivos estação-ano; 143 mudaram de coordenada em mais de 0,01° no período.


**Resumo da camada de leitura (INMET):** 13 anos de dados horários de todas as estações
automáticas, lidos e padronizados em `data/01_leitura/inmet/`. Próximo passo:
**`02_tratamento/02_tratamento_inmet.ipynb`**.